# Bronze Layer — Ingesting NJ Transit Data

This notebook ingests raw NJ Transit data into the `workspace.nj_transit_bronze` schema: train trip CSVs from Kaggle (loaded from a UC volume), rail stations and lines from the ArcGIS FeatureServer API, and hourly weather for NYC, Newark, and Trenton from the Open-Meteo archive API. 

In [0]:
# --- Imports: core Python, Spark, API ---
# Standard Python libs for tabular and file handling
import pandas as pd
import numpy as np
import json
import os
import shutil

# HTTP client: used for all API calls below
import requests

# PySpark: for DataFrame operations in the lakehouse
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("All imports loaded successfully.")

In [0]:
# --- Import Kaggle train trip CSVs into bronze layer ---
# Source: UC volume /Volumes/workspace/nj_transit_bronze/kaggle_data/
folder_path = "/Volumes/workspace/nj_transit_bronze/kaggle_data/"

df_raw_trips = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(folder_path)
)

df_raw_trips.show()  # Preview raw import (shows column names and types)

In [0]:
# --- Save raw transit trips table to bronze ---
df_raw_trips.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_bronze.transit_trips")

print("Delta table saved: workspace.nj_transit_bronze.transit_trips")

# Quick sanity check: row count
print(f"Row count: {spark.table('workspace.nj_transit_bronze.transit_trips').count()}")

In [0]:
# --- Import NJ Transit station data via ArcGIS API ---
# Endpoint returns GeoJSON features — each has properties (attributes) and geometry (location)
api_url_stations = 'https://services6.arcgis.com/M0t0HPE53pFK525U/arcgis/rest/services/NJTransit_Rail_Stations/FeatureServer/0/query'

params = {
    "where": "1=1",           # fetch all rows
    "outFields": "*",         # fetch all columns
    "f": "geojson",           # GeoJSON response
    "returnGeometry": "true"  # include geometry
}

response = requests.get(api_url_stations, params=params)
print(f"Status Code: {response.status_code}")

data = response.json()
features = data.get('features', [])

# Extract flat records for Spark DataFrame
records = [f['properties'] for f in features if 'properties' in f]
df_stations = spark.createDataFrame(pd.DataFrame(records))

In [0]:
# --- Save station table to bronze ---
df_stations.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_bronze.stations")

print("Delta table saved: workspace.nj_transit_bronze.transit_stations")

In [0]:
# --- Import NJ Transit rail lines from ArcGIS endpoint ---
# 14 separate endpoints, each returns GeoJSON records for a line or segment group
base_line_endpoint = 'https://services6.arcgis.com/M0t0HPE53pFK525U/arcgis/rest/services/NJTransit_Rail_Linea/FeatureServer'
records = []

for i in range(0, 14):  # Loop over 14 line endpoints
    current_endpoint = f"{base_line_endpoint}/{i}/query"
    params = {
        "where": "1=1",
        "outFields": "*",
        "f": "geojson",
        "returnGeometry": "true"
    }
    response = requests.get(current_endpoint, params=params)
    print(f"Status Code: {response.status_code}")
    data = response.json()
    features = data.get("features", [])
    for f in features:
        props = f.get("properties", {}) or {}
        geom = f.get("geometry", {}) or {}
        record = {
            **props,
            "geometry_type": geom.get("type"),
            "coordinates": str(geom.get("coordinates")),  # Serialize geometry
        }
        records.append(record)

# Convert line records to Spark DataFrame and save to bronze
df_lines = spark.createDataFrame(pd.DataFrame(records))
df_lines.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_bronze.lines")

In [0]:
# Preview the assembled lines DataFrame (shows geometry columns for all lines)
df_lines.show()

In [0]:
# --- Import hourly weather for NYC, Newark, Trenton from Open-Meteo ---
weather_url = "https://archive-api.open-meteo.com/v1/archive"

nec_locations = [
    {"region": "New York City (NY)", "latitude": 40.7128, "longitude": -74.0060},
    {"region": "Newark (NJ)", "latitude": 40.7357, "longitude": -74.1724},
    {"region": "Trenton (NJ)", "latitude": 40.2206, "longitude": -74.7597},
]

start_date = '2018-01-01'
end_date = '2020-12-31'

all_weather_list = []

for loc in nec_locations:
    # Query the archive API for each region separately
    params = {
        "latitude": loc['latitude'],
        "longitude": loc['longitude'],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,wind_speed_10m",
        "timezone": "America/New_York"
    }
    response = requests.get(weather_url, params=params)
    data = response.json()
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", [])
    precips = hourly.get("precipitation", [])
    winds = hourly.get("wind_speed_10m", [])
    for ti, te, pr, wi in zip(times, temps, precips, winds):
        all_weather_list.append({
            "region_name": loc["region"],
            "latitude": loc["latitude"],
            "longitude": loc["longitude"],
            "timestamp": ti,
            "temperature_2m_c": te,
            "precipitation_mm": pr,
            "wind_speed_kmh": wi,
        })

# Save weather records to bronze as Delta table
df_weather = spark.createDataFrame(pd.DataFrame(all_weather_list))
df_weather.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_bronze.weather")